# Aula 01 - Aquisição de Dados de Mercado: Coleta Confiável e Reproduzível

Módulo 08 AdmTech, Computação · Semana 1 · Sprint 1 · Prof. José Romualdo

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josercf/2026-2B-M08/blob/main/notebooks/aula01.ipynb)

## O que este notebook é

Laboratório executável da Aula 01, que alimenta o artefato A1 (Coletor e Processador de Dados de Mercado). Roda no Colab pelo link acima ou localmente a partir do repositório clonado. Sem rede, cada célula de carga usa o snapshot versionado em `dados/`.

## 1. A cesta e onde os dados moram

A cesta reúne 15 ativos negociados na B3: PETR4, VALE3, ITUB4, BBDC4, BBAS3, ABEV3, WEGE3, EGIE3, SUZB3, RENT3, B3SA3, MGLU3, BOVA11, IVVB11 e GOLD11. A janela vai de 01/01/2019 a 31/08/2026. O snapshot desses ativos, coletado em 17/09/2026, mora em `dados/`, e é para lá que este notebook cai quando não encontra o repositório clonado ao lado ou quando a rede está desligada.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DADOS_LOCAL = os.path.join("..", "dados")
DADOS_BRUTOS = "https://raw.githubusercontent.com/josercf/2026-2B-M08/main/dados/"
TICKERS = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "ABEV3", "WEGE3", "EGIE3",
           "SUZB3", "RENT3", "B3SA3", "MGLU3", "BOVA11", "IVVB11", "GOLD11"]
INICIO, FIM = "2019-01-01", "2026-08-31"


def ler_csv(nome, **kwargs):
    """Lê um CSV de dados/: do disco quando o repositório está clonado, da URL bruta no Colab."""
    local = os.path.join(DADOS_LOCAL, nome)
    return pd.read_csv(local if os.path.exists(local) else DADOS_BRUTOS + nome, **kwargs)

## 2. Um coletor que outra pessoa consegue rodar

Parametrizar tickers e datas em vez de fixá-los no código deixa o mesmo coletor servir o A1 de qualquer grupo, cada um com a própria cesta e a própria janela. A queda para o snapshot de `dados/` garante que o notebook termina mesmo sem rede, sem chave de API ou com o provedor fora do ar, e é o que a variável `M08_OFFLINE=1` força mesmo com rede disponível, para reproduzir localmente o que roda no CI. Um detalhe do `yfinance`: o parâmetro `end` é exclusivo, então o coletor soma um dia à data final antes de chamar a biblioteca, para incluir o último pregão pedido.

In [ ]:
def coletar(tickers, inicio, fim):
    """Close e Adj Close dos tickers entre inicio e fim, inclusive.

    Tenta o yfinance ao vivo; sem rede, sem yfinance ou com M08_OFFLINE=1, lê o
    snapshot versionado em dados/. Devolve (fechamento, ajustado, origem).
    """
    if os.environ.get("M08_OFFLINE") != "1":
        try:
            import yfinance as yf

            fim_exclusivo = (pd.Timestamp(fim) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
            bruto = yf.download([t + ".SA" for t in tickers], start=inicio, end=fim_exclusivo,
                                auto_adjust=False, progress=False, group_by="column",
                                threads=False, timeout=20)
            if not bruto.empty:
                renomear = lambda coluna: coluna.removesuffix(".SA")
                fechamento = bruto["Close"].rename(columns=renomear)[tickers]
                ajustado = bruto["Adj Close"].rename(columns=renomear)[tickers]
                return fechamento, ajustado, "yfinance, ao vivo"
        except Exception as erro:
            print("coleta ao vivo falhou, seguindo com o snapshot:", type(erro).__name__)
    fechamento = ler_csv("precos_fechamento.csv", index_col="data", parse_dates=["data"])
    ajustado = ler_csv("precos_ajustados.csv", index_col="data", parse_dates=["data"])
    return fechamento.loc[inicio:fim, tickers], ajustado.loc[inicio:fim, tickers], "snapshot de dados/"


fechamento, ajustado, origem = coletar(TICKERS, INICIO, FIM)
print("origem:", origem)
print("pregões:", len(fechamento), "de", fechamento.index[0].date(), "a", fechamento.index[-1].date())

## 3. Três visões do mesmo preço

O `Close` já vem dividido pelos fatores de desdobramento, grupamento e bonificação, e responde ao retorno de quem comprou e manteve a posição através desses eventos. O `Adj Close` desconta também os proventos, e responde ao retorno de quem reinvestiu cada dividendo e juro sobre capital próprio recebido. O preço negociado é o valor que apareceu na tela do pregão naquele dia, reconstruído multiplicando o `Close` pelos fatores dos eventos posteriores à data, e responde ao que o investidor efetivamente viu no pregão.

In [ ]:
eventos = ler_csv("eventos_desdobramento.csv", parse_dates=["data"])


def preco_negociado(fechamento, eventos, ticker):
    """Close multiplicado pelos fatores dos eventos posteriores a cada data."""
    serie = fechamento[ticker].dropna()
    fator = np.ones(len(serie))
    do_ticker = eventos[eventos["ticker"] == ticker]
    for data, f in zip(do_ticker["data"], do_ticker["fator"]):
        fator = fator * np.where(serie.index < data, f, 1.0)
    return serie * fator


visoes = pd.DataFrame({
    "negociado": preco_negociado(fechamento, eventos, "MGLU3"),
    "Close": fechamento["MGLU3"],
    "Adj Close": ajustado["MGLU3"],
})
eixo = visoes.plot(logy=True, figsize=(11, 4), title="MGLU3: três visões de preço")
eixo.set_ylabel("R$ por ação (escala log)")
plt.show()

In [ ]:
for ticker in ["B3SA3", "MGLU3"]:
    negociado = preco_negociado(fechamento, eventos, ticker).pct_change().dropna()
    close = fechamento[ticker].dropna().pct_change().dropna()
    dia = negociado.abs().idxmax()
    print("%s em %s: negociado %+.2f%%, Close %+.2f%%" % (ticker, dia.date(), 100 * negociado[dia], 100 * close[dia]))

### Pergunta para a dupla

Quatro perguntas de negócio, cada uma respondida por exatamente uma das três visões de preço. Decidam em dupla antes de abrir a resposta.

1. No pregão de maior variação do preço negociado acima, o que essa série mostra que o `Close` não mostra?
2. Por que o `Close` não sofre esse mesmo solavanco, mesmo sendo o mesmo evento de desdobramento ou grupamento?
3. Ignorando todo provento, qual série mede o que rendeu quem comprou a ação e apenas segurou?
4. Somando cada provento reinvestido, qual série mede o que essa mesma decisão rendeu?

<details>
<summary>Resposta</summary>

1. Preço negociado: reconstruído com os fatores dos eventos posteriores, é a única série que reproduz o valor exibido na tela do pregão naquele dia.
2. Close: já sai dividido pelos fatores de desdobramento e grupamento, então o evento não aparece como salto nessa série.
3. Close: mede o retorno de quem não reinvestiu proventos.
4. Adj Close: desconta os proventos, então mede o retorno de quem reinvestiu tudo.

</details>

In [ ]:
for ticker in ["PETR4", "BBAS3", "BOVA11"]:
    preco = fechamento[ticker].dropna()
    total = ajustado[ticker].dropna()
    print("%s: retorno de preço %+.2f%%, retorno total %+.2f%%"
          % (ticker, 100 * (preco.iloc[-1] / preco.iloc[0] - 1), 100 * (total.iloc[-1] / total.iloc[0] - 1)))

## 4. Ausências reais do provedor

Nem todo ativo tem cotação em todos os pregões da janela: um fundo listado depois do início da coleta fica sem preço até estrear, e o provedor às vezes falha em publicar a cotação de um pregão normal da B3. As duas próximas células medem quantos pregões faltam por ativo e comparam o que acontece com a série quando essas ausências são descartadas contra quando são preenchidas com o último preço conhecido.

In [ ]:
vazios = fechamento.isna().sum()
print(vazios[vazios > 0])
print("primeira cotação do GOLD11:", fechamento["GOLD11"].first_valid_index().date())
print("ativos sem cotação em 24/10/2025:", list(fechamento.columns[fechamento.loc["2025-10-24"].isna()]))

In [ ]:
def volatilidade_anual(precos):
    return precos.pct_change(fill_method=None).std() * np.sqrt(252)


descartar = ajustado.dropna()
preencher = ajustado.ffill()
comparacao = pd.DataFrame({
    "descartar (dropna)": volatilidade_anual(descartar),
    "preencher (ffill)": volatilidade_anual(preencher),
}).loc[["BOVA11", "IVVB11", "GOLD11", "PETR4", "VALE3"]]
print("pregões mantidos: dropna %d, ffill %d, de %d" % (len(descartar), len(preencher), len(ajustado)))
print("a série do dropna começa em", descartar.index[0].date())
(100 * comparacao).round(2)

## 5. Registre a decisão

Cada grupo grava um ADR com a escolha entre descartar as datas com ausência ou preencher para a frente, a partir do [modelo do professor](https://gist.github.com/josercf/fc714afe20773401dc9d56f65174b441). O ADR precisa citar o número que muda com a escolha, medido na célula anterior: a volatilidade anual de cada ativo sob os dois tratamentos.

## 6. Desafio para o A1

a. Trocar a cesta pelos tickers do grupo e rodar `coletar` com a janela escolhida pelo grupo.
b. Escrever no `SPEC.md` três casos de teste com o resultado esperado, sendo um deles com os dados do próprio grupo.
c. Completar a célula seguinte com as verificações que esses três casos de teste exigem.

In [ ]:
# Verificações mínimas de reprodutibilidade. Acrescente as do SPEC.md do grupo.
assert fechamento.index.is_monotonic_increasing, "datas fora de ordem"
assert not fechamento.index.duplicated().any(), "data repetida"
assert (fechamento.min() > 0).all(), "preço não positivo"  # min() ignora as ausências
print("verificações ok")